In [2]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip,DeltaTable
from pyspark.sql.functions import input_file_name,regexp_extract,current_timestamp

In [3]:
import os
import json
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
spark_builder = (
    SparkSession.builder
    .appName("IngestingtoBronzelayer")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(spark_builder).getOrCreate()

spark.sparkContext.setLogLevel("ERROR")


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e9413089-0196-46eb-abde-018e8026daaa;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 229ms :: artifacts dl 12ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |  

In [20]:
import json
import os

config_path = "/root/.config/kaggle/kaggle_login.json"

with open(config_path) as f:
    secrets = json.load(f)

os.environ["KAGGLE_USERNAME"] = secrets["kaggle_username"]
os.environ["KAGGLE_KEY"] = secrets["kaggle_key"]



In [16]:
import os
#Run only one time for downloading once done then it's not required 
download_path = "/opt/spark/data/raw/batch_1"

if not os.path.exists(download_path):
    !kaggle datasets download -d osamahosamabdellatif/high-quality-invoice-images-for-ocr \
      -p /opt/spark/data/raw --unzip
else:
    print("Dataset already downloaded.")


Dataset already downloaded.


In [4]:
batch1_path = "/opt/spark/data/raw/batch_1/batch_1"

# Read all CSV files in the folder into a single Spark DataFrame
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("multiline", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("inferSchema", "true") \
    .load(f"{batch1_path}/batch1_3.csv")   # change path

df.show(5)

+---------------+--------------------+--------------------+
|      File Name|           Json Data|          OCRed Text|
+---------------+--------------------+--------------------+
|batch1-1142.jpg|\n{\n  "invoice":...|Invoice no: 96556...|
|batch1-1147.jpg|\n{\n  "invoice":...|Invoice no: 95894...|
|batch1-1080.jpg|\n{\n  "invoice":...|Invoice no: 17585...|
|batch1-1146.jpg|\n{\n  "invoice":...|Invoice no: 27702...|
|batch1-1090.jpg|\n{\n  "invoice":...|Invoice no: 34310...|
+---------------+--------------------+--------------------+
only showing top 5 rows



In [5]:
from pyspark.sql.functions import (
    input_file_name,
    regexp_extract,
    current_timestamp
)

df_bronze = (df.withColumn("source_file",regexp_extract(input_file_name(), r"([^/]+$)", 1))
    .withColumn("ingestion_ts", current_timestamp())
    .withColumnRenamed("File Name","file_name")
    .withColumnRenamed("Json Data","json_data")
    .withColumnRenamed("OCRed Text","ocred_text")
)


In [6]:
df_bronze.show(5)

+---------------+--------------------+--------------------+------------+--------------------+
|      file_name|           json_data|          ocred_text| source_file|        ingestion_ts|
+---------------+--------------------+--------------------+------------+--------------------+
|batch1-1142.jpg|\n{\n  "invoice":...|Invoice no: 96556...|batch1_3.csv|2026-02-09 09:14:...|
|batch1-1147.jpg|\n{\n  "invoice":...|Invoice no: 95894...|batch1_3.csv|2026-02-09 09:14:...|
|batch1-1080.jpg|\n{\n  "invoice":...|Invoice no: 17585...|batch1_3.csv|2026-02-09 09:14:...|
|batch1-1146.jpg|\n{\n  "invoice":...|Invoice no: 27702...|batch1_3.csv|2026-02-09 09:14:...|
|batch1-1090.jpg|\n{\n  "invoice":...|Invoice no: 34310...|batch1_3.csv|2026-02-09 09:14:...|
+---------------+--------------------+--------------------+------------+--------------------+
only showing top 5 rows



In [7]:
bronze_path = '/opt/spark/data/bronze/invoices_raw'

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS invoices_ocr
    (
        file_name string,
        json_data string,
        ocred_text string,
        source_file string,
        ingestion_ts timestamp
    )
    USING DELTA
    LOCATION '{bronze_path}'
""")


DataFrame[]

In [8]:
df_bronze.write.format("delta").mode("append").option("mergeSchema",True).save(bronze_path)

In [11]:
import os
import shutil

# Paths in your container
raw_path = "/opt/spark/data/raw/batch_1/batch_1"
processed_path = "/opt/spark/data/processed/batch_1/"

# Make sure processed folder exists
os.makedirs(processed_path, exist_ok=True)

# List all files in raw folder
for file_name in os.listdir(raw_path):
    if file_name.endswith("_3.csv"):  # move only CSVs
        src = os.path.join(raw_path, file_name)
        dst = os.path.join(processed_path, file_name)
        shutil.move(src, dst)

print(f"CSV files moved from {raw_path} to {processed_path}")


CSV files moved from /opt/spark/data/raw/batch_1/batch_1 to /opt/spark/data/processed/batch_1/
